Title:
Train Delay Prediction for Operational Optimization – Indian Railways

Objective:
This analysis explores delay patterns in Indian Railways and demonstrates how predictive signals can support operational prioritization, resource allocation, and improved passenger communication.

Data Overview

- Dataset: ~100,000 simulated station-arrival records
- Target variable: Arrival_Delay_min

- Key feature groups:
  - Scheduling & route: Distance, stops, travel time
  - Operational: signal failure, maintenance, engine breakdown, crew change
  - Network effects: previous train delay
  - Passenger experience: passenger load, weather conditions

In [1]:
import pandas as pd

# Load the dataset (file is in Colab working directory)
df = pd.read_csv("stratquest_dataset.csv")

# Basic inspection
print("Dataset shape (rows, columns):", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'stratquest_dataset.csv'

Data Preprocessing

- Removed index column
- Converted timestamps to datetime
- Handled missing values
- Normalized passenger load
- Encoded operational flags




In [ ]:
# -----------------------------
# Data Preprocessing
# -----------------------------

# 1. Drop useless index column if present
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# 2. Convert timestamp columns to datetime
datetime_cols = [
    'Scheduled_Departure',
    'Scheduled_Arrival',
    'Actual_Departure',
    'Actual_Arrival'
]

for col in datetime_cols:
    df[col] = pd.to_datetime(df[col])

# 3. Clip Passenger_Load_pct to valid range
df['Passenger_Load_pct'] = df['Passenger_Load_pct'].clip(0, 100)

# 4. Fill missing Previous_Train_Delay_min with 0
df['Previous_Train_Delay_min'] = df['Previous_Train_Delay_min'].fillna(0)

# 5. Show missing values after cleaning
print("Missing values after cleaning:")
df.isna().sum()


4.1 Arrival Delay Distribution

- Most trains experience minor delays
- A smaller subset experiences large delays, driving cascading effects
- ~XX% of trains are delayed more than 15 minutes

In [ ]:
import matplotlib.pyplot as plt

# Histogram of Arrival Delay
plt.figure(figsize=(8,5))
plt.hist(df['Arrival_Delay_min'], bins=50)
plt.xlabel('Arrival Delay (minutes)')
plt.ylabel('Number of Trains')
plt.title('Distribution of Train Arrival Delays')
plt.show()

# Percentage of trains delayed more than 15 minutes
delayed_pct = (df['Arrival_Delay_min'] > 15).mean() * 100
print(f"Percentage of trains delayed more than 15 minutes: {delayed_pct:.2f}%")


4.2 Delay Propagation

- Strong positive relationship between previous train delay and current arrival delay
- Confirms cascading delay behavior across the network

In [ ]:
import matplotlib.pyplot as plt

# Scatter plot: Previous train delay vs current arrival delay
plt.figure(figsize=(7,5))
plt.scatter(
    df['Previous_Train_Delay_min'],
    df['Arrival_Delay_min'],
    alpha=0.3
)
plt.xlabel('Previous Train Delay (minutes)')
plt.ylabel('Current Train Arrival Delay (minutes)')
plt.title('Delay Propagation Effect')
plt.show()

# Correlation value
corr = df['Previous_Train_Delay_min'].corr(df['Arrival_Delay_min'])
print(f"Correlation between previous train delay and arrival delay: {corr:.2f}")


4.3 Operational Disruptions

- Signal failures, track maintenance, and engine breakdowns cause disproportionately high delays
- These events should be prioritized for rapid intervention

In [ ]:
# Average delay by operational disruption flags

ops_cols = ['Signal_Failure', 'Track_Maintenance', 'Engine_Breakdown']

for col in ops_cols:
    print(f"\nImpact of {col}:")
    print(
        df.groupby(col)['Arrival_Delay_min']
        .mean()
        .round(2)
    )


4.4 Passenger Load & Assets

- Higher passenger load increases delay risk
- Locomotive type shows asset-level delay differences
- Crew change and number of stops add operational friction

In [ ]:
# Average arrival delay by locomotive type
loco_delay = (
    df.groupby('Loco_Type')['Arrival_Delay_min']
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

print("Average arrival delay by locomotive type:")
print(loco_delay)


In [ ]:
# Convert Yes/No columns to 1/0
binary_cols = [
    'Track_Maintenance',
    'Signal_Failure',
    'Engine_Breakdown',
    'Crew_Change'
]

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})


Feature Importance & Model Summary

Model Used
- Random Forest Regressor
- Chosen for robustness and interpretability
- Focus on feature ranking, not perfect prediction

Feature Importance Insights
- Passenger_Load_pct is the strongest predictor, reflecting congestion and dwell-time sensitivity.
- Distance_km and Number_of_Stops capture route complexity and exposure to disruptions.
- Previous_Train_Delay_min confirms delay propagation effects across the network.
- Engine_Breakdown and Track_Maintenance drive high-delay events despite lower frequency.
- Crew_Change has limited standalone impact but may compound delays during congestion.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import pandas as pd

# Select important features
features = [
    'Previous_Train_Delay_min',
    'Passenger_Load_pct',
    'Track_Maintenance',
    'Signal_Failure',
    'Engine_Breakdown',
    'Crew_Change',
    'Number_of_Stops',
    'Distance_km'
]

X = df[features]
y = df['Arrival_Delay_min']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Feature importance
importance = pd.Series(
    model.feature_importances_,
    index=features
).sort_values(ascending=False)

print("Feature importance:")
print(importance)


Strategic Recommendations

Operational Optimization
- Prioritize intervention on trains following heavily delayed services
- Rapid response to signal failures, maintenance, and breakdown flags

Resource Allocation
- Assign reliable locomotive types to high passenger-load services
- Minimize crew changes during peak congestion windows

Passenger Experience
- Use delay risk signals (>15 min) for proactive ETA updates
- Support contingency planning and compensation readiness

Conclusion

This analysis demonstrates how delay prediction combined with operational signals can reduce cascading delays, improve on-time performance, and enhance passenger experience through targeted, data-driven interventions.